In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
items = pd.read_csv(
    "../data/raw/olist_order_items_dataset.csv"
)

items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [3]:
seller_count = (
    items.groupby("order_id")["seller_id"]
    .nunique()
    .reset_index(name="seller_count")
)

multi_seller_orders = seller_count[
    seller_count["seller_count"] > 1
]

print("Total orders:", len(seller_count))

print("Multi-seller orders:", len(multi_seller_orders))

print(
    "Percentage:",
    round(
        len(multi_seller_orders) / len(seller_count) * 100,
        2
    ),
    "%"
)

Total orders: 98666
Multi-seller orders: 1278
Percentage: 1.3 %


In [4]:
deadline_count = (
    items.groupby("order_id")["shipping_limit_date"]
    .nunique()
    .reset_index(name="deadline_count")
)

multiple_deadlines = deadline_count[
    deadline_count["deadline_count"] > 1
]

print("Orders with multiple shipping deadlines:",
      len(multiple_deadlines))

Orders with multiple shipping deadlines: 336


In [5]:
items[items["order_id"].isin(
    multiple_deadlines["order_id"].head(5)
)].sort_values("order_id")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
788,01cce1175ac3c4a450e3a0f856d02734,1,415dfa57292b8b7360d3f4cf2f9bff06,9c7fd21e0aa159b20da593371f08ff7c,2018-07-25 20:43:14,60.00,25.24
789,01cce1175ac3c4a450e3a0f856d02734,2,9d0aa87e8df1bdbe0f79353520a2d538,0bae85eb84b9fb3bd773911e89288d54,2018-07-23 20:43:14,36.23,2.27
1079,0282e83acfe3e7aaf60c871a4a9aca61,1,3faadd91c89ff779e6cd1fd09a46ee39,db4350fd57ae30082dec7acbaacc17f9,2018-05-16 20:57:02,58.00,31.12
1080,0282e83acfe3e7aaf60c871a4a9aca61,2,7d0e0214a0852cf0c90032c7992cf27f,2a6fbeaa7ab9e5d59a35841a8da71ce4,2018-05-17 20:57:02,65.00,9.33
1200,02c9fa29186a8d00b3e73d3837ea40e5,1,f63e5eb82736ac58d9d286f64fe5c44f,08ad4ac1388e4420ca531c3edfc46198,2018-08-22 19:09:45,25.99,18.25
1201,02c9fa29186a8d00b3e73d3837ea40e5,2,caa526308abc9a77d0c56869139dbae9,6061155addc1e54b4cfb51c1c2a32ad8,2018-08-21 19:09:45,19.00,18.25
1249,02e405aa2667a116637cdb4affaf93c0,1,013ee64977aaa6b2b25475095162e0e9,640e21a7d01df7614a3b4923e990d40c,2018-04-23 23:51:37,69.99,31.87
1250,02e405aa2667a116637cdb4affaf93c0,2,f6fa93875f4a7dac03e0a22143f4d815,391fc6631aebcf3004804e51b40bcf1e,2018-04-26 23:51:37,49.95,10.52
1713,03ecec245220b63fd7f68c1737ba99ba,1,d8cabd83757ecf48a7d2628dee9135c1,fe2032dab1a61af8794248c8196565c9,2018-06-13 20:23:08,189.00,58.78
1714,03ecec245220b63fd7f68c1737ba99ba,2,059ca09d32720d9adce3ae12e181339d,620c87c171fb2a6dd6e8bb4dec959fc6,2018-06-15 20:23:08,109.90,18.05


In [6]:
order_analysis = (
    items.groupby("order_id")
    .agg(
        seller_count=("seller_id", "nunique"),
        deadline_count=("shipping_limit_date", "nunique"),
        item_count=("order_item_id", "count")
    )
    .reset_index()
)

# Orders with multiple sellers AND multiple deadlines
both_multiple = order_analysis[
    (order_analysis["seller_count"] > 1) &
    (order_analysis["deadline_count"] > 1)
]

# Orders with multiple sellers but one deadline
multiple_sellers_one_deadline = order_analysis[
    (order_analysis["seller_count"] > 1) &
    (order_analysis["deadline_count"] == 1)
]

print("Multiple sellers AND deadlines:", len(both_multiple))

print("Multiple sellers, one deadline:",
      len(multiple_sellers_one_deadline))

Multiple sellers AND deadlines: 336
Multiple sellers, one deadline: 942


In [7]:
print(
    pd.crosstab(
        order_analysis["seller_count"] > 1,
        order_analysis["deadline_count"] > 1,
        rownames=["Multiple Sellers"],
        colnames=["Multiple Deadlines"]
    )
)

Multiple Deadlines  False  True 
Multiple Sellers                
False               97388      0
True                  942    336


## Initial Findings

- Total unique orders: 98,666
- Multi-seller orders: 1,278 (1.30%)
- Orders with multiple shipping deadlines: 336

### Observations
- An order can contain products from multiple sellers.
- Different items can have different shipping deadlines.
- Shipping deadlines should initially be analyzed at item level.
- Carrier-specific performance cannot be measured using the order items dataset alone.

In [8]:


orders = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv"
)

print(orders.shape)
print(orders.columns.tolist())

orders.head()

(99441, 8)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [9]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [10]:
orders.isnull().sum()


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [11]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Timestamp conversion completed!")

Timestamp conversion completed!


In [12]:
delivered = orders[
    orders["order_status"] == "delivered"
].copy()

print("Total delivered orders:", len(delivered))

Total delivered orders: 96478


In [13]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Timestamp conversion completed!")

Timestamp conversion completed!


In [14]:
delivered = orders[
    orders["order_status"] == "delivered"
].copy()

print("Total delivered orders:", len(delivered))
print("\nMissing timestamps in delivered orders:")
print(delivered[date_columns].isnull().sum())

Total delivered orders: 96478

Missing timestamps in delivered orders:
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


In [15]:
delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"] -
    delivered["order_purchase_timestamp"]
).dt.total_seconds() / 86400

delivered["delivery_delay_days"] = (
    delivered["order_delivered_customer_date"] -
    delivered["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

delivered[[
    "order_id",
    "delivery_days",
    "delivery_delay_days"
]].head()

,order_id,delivery_days,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,-7.107488
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,-5.355729
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,-17.245498
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,-12.980069
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,-9.238171


In [16]:
delivery_data = delivered.dropna(
    subset=["order_delivered_customer_date"]
).copy()

print("Records available for delivery analysis:",
      len(delivery_data))

Records available for delivery analysis: 96470


In [17]:
delivery_data["delivery_days"] = (
    delivery_data["order_delivered_customer_date"] -
    delivery_data["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [18]:
delivery_data["delivery_delay_days"] = (
    delivery_data["order_delivered_customer_date"] -
    delivery_data["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

delivery_data[[
    "order_id",
    "delivery_days",
    "delivery_delay_days"
]].head(10)

,order_id,delivery_days,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,-7.107488
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,-5.355729
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,-17.245498
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,-12.980069
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,-9.238171
5,a4591c265e18cb1dcee52889e2d8acc3,16.542245,-5.543113
7,6514b8ad8028c9f2cc2374ded245783f,9.989826,-11.461215
8,76c6e866289321a7c93b82b54852dc33,9.818762,-31.410995
9,e69bfb5eb88e0ed6a785585b27e16dbf,18.221852,-6.281597
10,e6ce16cb79ec1d90b1da9085a6118aeb,12.650937,-8.528808


In [19]:
print("Average delivery time:",
      round(delivery_data["delivery_days"].mean(), 2), "days")

print("Median delivery time:",
      round(delivery_data["delivery_days"].median(), 2), "days")

print("Average delivery delay:",
      round(delivery_data["delivery_delay_days"].mean(), 2), "days")

Average delivery time: 12.56 days
Median delivery time: 10.22 days
Average delivery delay: -11.18 days


In [20]:
delivery_data["delivery_delay_days"] = (
    delivery_data["order_delivered_customer_date"].dt.normalize()
    -
    delivery_data["order_estimated_delivery_date"].dt.normalize()
).dt.days

In [21]:
early = (delivery_data["delivery_delay_days"] < 0).sum()

on_time = (delivery_data["delivery_delay_days"] == 0).sum()

late = (delivery_data["delivery_delay_days"] > 0).sum()

print("Early deliveries:", early)
print("On-time deliveries:", on_time)
print("Late deliveries:", late)

Early deliveries: 88644
On-time deliveries: 1292
Late deliveries: 6534


In [22]:
delivery_data["delivery_delay_days"] = (
    delivery_data["order_delivered_customer_date"].dt.normalize()
    -
    delivery_data["order_estimated_delivery_date"].dt.normalize()
).dt.days

In [23]:
early = (delivery_data["delivery_delay_days"] < 0).sum()

on_time = (delivery_data["delivery_delay_days"] == 0).sum()

late = (delivery_data["delivery_delay_days"] > 0).sum()

total = len(delivery_data)

print("Early deliveries:", early)
print("On-time deliveries:", on_time)
print("Late deliveries:", late)

print("\nDelivery percentages:")

print("Early:", round(early / total * 100, 2), "%")
print("On-time:", round(on_time / total * 100, 2), "%")
print("Late:", round(late / total * 100, 2), "%")

Early deliveries: 88644
On-time deliveries: 1292
Late deliveries: 6534

Delivery percentages:
Early: 91.89 %
On-time: 1.34 %
Late: 6.77 %


In [24]:
print("Average delivery time:",
      round(delivery_data["delivery_days"].mean(), 2), "days")

print("Median delivery time:",
      round(delivery_data["delivery_days"].median(), 2), "days")

print("Average delivery delay:",
      round(delivery_data["delivery_delay_days"].mean(), 2), "days")

Average delivery time: 12.56 days
Median delivery time: 10.22 days
Average delivery delay: -11.88 days


---
# Day 1: Interim Dataset Loading and Eligible Order Filtering

**Objective:** Load `data/interim/logistics_analytical_dataset.csv`, inspect dataset dimensions, relevant column definitions, missing values, and filter eligible orders using `is_eligible_otd == True`.

> **Note:** The cells above this section are exploratory background work using raw datasets. The five-day analysis plan begins here, using the pre-processed interim analytical dataset.

In [25]:
# Load the pre-processed analytical dataset
df = pd.read_csv('../data/interim/logistics_analytical_dataset.csv')

print(f'Dataset loaded successfully!')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

Dataset loaded successfully!
Shape: 99,441 rows x 39 columns


In [26]:
# Inspect key columns relevant to this carrier analysis
key_cols = [
    'order_id', 'seller_id',
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'is_eligible_otd', 'order_processing_time_days',
    'carrier_transit_time_days', 'total_delivery_duration_days',
    'promised_delivery_duration_days', 'delay_days', 'delivery_outcome'
]

print('Column dtypes:')
print(df[key_cols].dtypes.to_string())
print('\nNull counts:')
print(df[key_cols].isnull().sum().to_string())

Column dtypes:
order_id                               str
seller_id                              str
order_purchase_timestamp               str
order_approved_at                      str
order_delivered_carrier_date           str
order_delivered_customer_date          str
is_eligible_otd                       bool
order_processing_time_days         float64
carrier_transit_time_days          float64
total_delivery_duration_days       float64
promised_delivery_duration_days    float64
delay_days                         float64
delivery_outcome                       str

Null counts:
order_id                              0
seller_id                           775
order_purchase_timestamp              0
order_approved_at                   160
order_delivered_carrier_date       1783
order_delivered_customer_date      2965
is_eligible_otd                       0
order_processing_time_days         1972
carrier_transit_time_days          3160
total_delivery_duration_days       2972
promised_del

In [27]:
# Filter eligible orders: delivered orders with valid timestamps
# is_eligible_otd == True: order_status='delivered', no invalid timestamps
valid = df[df['is_eligible_otd'] == True].copy()

# Create named analysis columns (per Day 2 plan)
valid['processing_days'] = valid['order_processing_time_days']
valid['transit_days']    = valid['carrier_transit_time_days']

print(f'Total orders in dataset  : {len(df):,}')
print(f'Eligible orders (OTD set): {len(valid):,}')
print(f'Ineligible (excluded)    : {len(df) - len(valid):,}')
print(f'\nDelivery outcome breakdown (eligible orders):')
print(valid['delivery_outcome'].value_counts().to_string())
print(f"\nOn-time delivery rate: {(valid['delivery_outcome'] == 'on_time').mean()*100:.1f}%")

Total orders in dataset  : 99,441
Eligible orders (OTD set): 96,281
Ineligible (excluded)    : 3,160

Delivery outcome breakdown (eligible orders):
delivery_outcome
on_time    88459
delayed     7822

On-time delivery rate: 91.9%


### Day 1 Findings

| Metric | Value |
|--------|-------|
| Total orders in dataset | 99,441 |
| **Eligible orders (OTD set)** | **96,281** |
| Ineligible / excluded | 3,160 |
| On-time deliveries | 88,459 (91.9%) |
| Delayed deliveries | 7,822 (8.1%) |

**Key observations:**
- Ineligible orders: canceled, non-delivered, or missing timestamps — correctly excluded.
- `order_processing_time_days` and `carrier_transit_time_days` have **zero nulls** in the eligible set.
- `delivery_outcome` is pre-computed: `on_time` or `delayed` relative to `order_estimated_delivery_date`.
- `shipping_limit_date` is not in the interim dataset — it will be joined from the raw items file in Day 3.